# Marketing Mix Modeling with Google Meridian — Lecture Notebook

**A reproducible, synthetic-data introduction to attribution, incrementality, Bayesian MMM, and scenario planning.**

This is the companion notebook to *Marketing Measurement: From Attribution to Influence* (October 2026).

**What the notebook measures:** the current model targets **weekly revenue** in USD. Media impressions are the execution variables and media spend is supplied for economic reporting. This is **not a conversion-count model** and should not be described as measuring cost per acquired customer.

**Scope:** national weekly synthetic e-commerce data (156 weeks, Google Search, Meta and YouTube), consumer demand control, promotion treatment. All results are estimates under model assumptions—not observed causal ground truth. The dataset is fictional and contains no customer information.

**Start here:** read the [README](../README.md) for environment setup and the [data dictionary](../docs/data_dictionary.md). This notebook is designed for live teaching; posterior sampling is disabled by default because it can be time-consuming.


## 1. Our business question (2 minutes)
A retailer advertises through **Google Search, Meta, and YouTube**. How much incremental revenue does each channel produce? Are there diminishing returns?

Conceptually: $Revenue_t = Baseline_t + Search_t + Meta_t + YouTube_t + Controls_t + \epsilon_t$.

`revenue` is the target, `*_impressions` are the **media execution metrics**, `*_spend` are the corresponding costs, `consumer_demand_index` is an outside control, and `promotion` is a company-controlled non-media treatment.

## 2. Load synthetic data (3 minutes)
The CSV is packaged in the same folder as this notebook. All figures are synthetic. We intentionally include seasonality, promotions, noise, advertising carryover, and saturation in the *data generator*.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = Path("../data/synthetic_weekly_mmm.csv")
df = pd.read_csv(CSV)
df["week"] = pd.to_datetime(df["week"])
assert len(df) == 156
assert not df.isna().any().any()
display(df.head())
print(f"Rows: {len(df)} | Dates: {df.week.min().date()} to {df.week.max().date()}")

**Teaching prompt:** Is a correlation between Google spend and revenue automatically proof that Google caused the revenue? **No.** Promotions, seasonality, and demand can move both spending and revenue.

## 3. Explore sales and spend (4 minutes)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(df["week"], df["revenue"] / 1e6)
axs[0].set_ylabel("Revenue ($M)")
for channel, label in [("google_search_spend", "Search"), ("meta_spend", "Meta"), ("youtube_spend", "YouTube")]:
    axs[1].plot(df["week"], df[channel] / 1e3, label=label)
axs[1].set_ylabel("Weekly spend ($K)")
axs[1].legend(ncol=3)
axs[1].set_xlabel("Week")
fig.suptitle("Our simulated retailer: revenue and media spend")
plt.tight_layout()
plt.show()

In [ ]:
print("Mean weekly revenue: ${:,.0f}".format(df.revenue.mean()))
print("Paid media spend by channel, 156 weeks:")
display(df[["google_search_spend","meta_spend","youtube_spend"]].sum().rename("Spend ($)").to_frame())

## 4. Intuition: carryover and diminishing returns (3 minutes)

**Adstock:** advertising can have effects beyond the week in which it appears. A simple example is $A_t = X_t + \lambda A_{t-1}$.

**Saturation:** at very high exposure, another impression is usually less productive than the first. A teaching example is $f(A)=A/(A+h)$. Meridian uses its own parameterized adstock and Hill saturation formulations; this is only a conceptual illustration.

In [ ]:
exposures = np.array([0, 0, 10, 0, 0, 0, 0, 0], dtype=float)
carryover = np.zeros_like(exposures)
for i, x in enumerate(exposures):
    carryover[i] = x + (0.55 * carryover[i-1] if i else 0)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(exposures)), exposures, alpha=.45, label="Direct exposure")
ax.plot(range(len(exposures)), carryover, marker="o", label="Adstock example")
ax.set(xlabel="Week", ylabel="Illustrative media units")
ax.legend(); plt.show()

## 5. Load the dataset into Google Meridian (3 minutes)

The **DataFrameInputDataBuilder** maps our wide-format table into the expected Meridian data structures. This is a **national model**, so there is no `geo` or population column. `revenue` is already monetary, so no `revenue_per_kpi` is needed.

In [ ]:
import meridian
from meridian.data import data_frame_input_data_builder
from meridian.model import model, spec
from meridian.analysis import visualizer, summarizer

print("Meridian imported successfully")
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="revenue",
    default_kpi_column="revenue",
    default_time_column="week",
    default_media_time_column="week"
)
builder.currency_code = "USD"

builder = (builder
    .with_kpi(df)
    .with_controls(df, control_cols=["consumer_demand_index"])
    .with_non_media_treatments(df, non_media_treatment_cols=["promotion"])
)

channels = ["Google Search", "Meta", "YouTube"]
builder = builder.with_media(
    df,
    media_cols=["google_search_impressions", "meta_impressions", "youtube_impressions"],
    media_spend_cols=["google_search_spend", "meta_spend", "youtube_spend"],
    media_channels=channels,
)
input_data = builder.build()
print("Meridian InputData successfully created")

## 6. Configure a simple Meridian model (2 minutes)
- `max_lag=4` allows up to 4 weeks of media carryover.
- `knots=12` uses a relatively compact time-varying intercept for trend and seasonality. This is a teaching choice, not a general recommendation.
- We begin with Meridian's default prior distributions.

**Important:** A Bayesian posterior only makes sense relative to its model assumptions and priors.

In [ ]:
model_spec = spec.ModelSpec(max_lag=4, knots=12)
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)
print("Model initialized. Next step: posterior sampling.")

## 7. Posterior inference — run BEFORE the lecture

**This is the slow cell.** Meridian uses NUTS MCMC and may take substantial time on a Mac CPU. Run in advance and save the notebook outputs. Do not try to fit during a live 20-minute demo unless you have already benchmarked the environment.

For a teaching run, four chains with a few hundred retained draws is a *starting point*, not assurance of convergence. For research or real business decisions, increase the sampling budget and run diagnostics thoroughly.

Set `RUN_MODEL=True` when you want to fit. It is deliberately `False` initially, so students can execute all introductory cells without launching a long computation.

In [ ]:
RUN_MODEL = False  # Change to True to sample; this may take time, especially on CPU
if RUN_MODEL:
    mmm.sample_prior(300, seed=2026)
    mmm.sample_posterior(n_chains=4, n_adapt=500, n_burnin=0, n_keep=500, seed=2026)
    print("Posterior sampling complete")
else:
    print("Sampling skipped. Set RUN_MODEL=True, then rerun this cell before the lecture.")

## 8. Inspect the results (present after fitting)

The following code is guarded by `RUN_MODEL` so a student can run through data preparation without fitting. **Do not interpret ROI or plots before checking convergence and model fit.**

For a confident production analysis, investigate R-hat, effective sample size, divergences, posterior predictive fit, sensitivity to priors, identifiability, and out-of-sample checks.

In [ ]:
if RUN_MODEL:
    diagnostics = visualizer.ModelDiagnostics(mmm)
    display(diagnostics.plot_rhat_boxplot())
    fit = visualizer.ModelFit(mmm)
    display(fit.plot_model_fit())
else:
    print("First fit the model in section 7 to display diagnostics and fit.")

In [ ]:
if RUN_MODEL:
    summary = summarizer.Summarizer(mmm)
    summary.output_model_results_summary(
        "meridian_results.html", ".",
        df.week.min().strftime("%Y-%m-%d"),
        df.week.max().strftime("%Y-%m-%d"),
    )
    print("Saved: meridian_results.html")
else:
    print("The report is generated after MCMC sampling, not before.")

## 9. Bayesian vs. frequentist discussion

Both approaches can use regression, adstock and saturation. Frequentist inference typically treats unknown coefficients as fixed parameters and studies estimator properties across repeated samples; Bayesian inference combines a likelihood with prior distributions to estimate a posterior distribution.

**Why Bayesian in this demonstration?** It offers direct posterior uncertainty statements and a coherent way to incorporate credible lift-experiment information. Neither approach automatically identifies confounders or guarantees causal estimates.

See [Bayesian vs. frequentist: one-page guide](../docs/bayesian_vs_frequentist.md) for a coin-toss illustration and MMM examples.


## 10. Broader discussion: the future of marketing measurement

Privacy-aware aggregate measurement, experiments combined with MMM, better causal reasoning, and trustworthy AI-mediated customer experiences. The traditional *4Ps → 7Ps → proposed 7Gs* marketing-strategy frameworks are discussed separately in [our reading guide](../docs/marketing_4p_7p_7g.md); they are not the statistical MMM method implemented here.


## 11. Class discussion

1. If a channel has the highest historical contribution, does that prove it has the highest marginal response?
2. Does an R² of 0.93 prove the estimated media effects are causal?
3. What changes if we increase YouTube spending without reducing another channel's spend?
4. What evidence would convince you to move $1 million between channels?


## References (official Google Meridian documentation)
- [Getting Started Demo](https://developers.google.com/meridian/notebook/meridian-getting-started)
- [Load national-level data](https://developers.google.com/meridian/docs/user-guide/load-national-data)
- [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model)
- [Run the model](https://developers.google.com/meridian/docs/user-guide/run-model)
- [Model specification](https://developers.google.com/meridian/docs/advanced-modeling/model-spec)

**Reproducibility note:** built against documented Meridian API in October 2026; actual execution of MCMC must be tested locally in your `meridian-mmm` environment. This notebook intentionally contains no precomputed Meridian posterior results.

## 12. Interactive marketing budget lab — per-channel saturation and spend scenarios

**Run this section after sampling the posterior, using the same fitted `mmm` object.** You do **not** need to refit Meridian when changing a budget.

There are two different charts:

- **Hill curves:** exposure (e.g., impressions) vs a *relative* saturation effect, before translating it into a monetary budget.
- **Spend-response curves:** dollars spent vs incremental revenue predicted by the fitted Bayesian MMM, under Meridian's fixed historical media flighting and cost-per-media-unit assumptions.

**KPI caution:** The original starter notebook models `revenue`, so its scenario units are **USD revenue**, even if `use_kpi=True`. To see incremental *conversions* you must first fit a model whose actual `kpi` column is total first-party conversions. Merely relabeling a revenue model will not convert it.

In [ ]:
# Install ipywidgets into this same conda environment if needed:
# %pip install ipywidgets
# Restart the kernel only if the install requests it (then rerun the notebook).

from IPython.display import display
from meridian.analysis import visualizer, optimizer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "mmm" not in globals() or getattr(mmm.inference_data, "posterior", None) is None:
    raise RuntimeError("Fit `mmm.sample_posterior(...)` first; an HTML report alone is not enough.")

# Revenue KPI: omit use_kpi=True to avoid redundant-setting warnings.
media_effects_kpi = visualizer.MediaEffects(mmm)

print("MERIDIAN: HILL SATURATION CURVES")
hill_charts = media_effects_kpi.plot_hill_curves(include_prior=False, include_ci=True)
for kind, chart in hill_charts.items():
    print(f"Channel type: {kind}")
    display(chart)

print("MERIDIAN: SPEND / INCREMENTAL REVENUE RESPONSE CURVES")
display(media_effects_kpi.plot_response_curves(plot_separately=True, include_ci=True))

### 12a. Precompute posterior spend-response curves once

After this cell completes, interacting with the sliders uses interpolation on **Meridian's actual fitted posterior response curves**. No MCMC reruns occur. Interpolation is meant for *teaching scenario exploration* and should not be mistaken for exact posterior re-evaluation at every arbitrary budget.

For a model with three paid channels, the slider range will be limited to the spend amounts covered by Meridian's curve grid. Those estimates can still involve extrapolation relative to historically observed spend, so we should not interpret them as guaranteed future effects.

In [ ]:
# IMPORTANT: May take time on a Mac CPU, but runs once, not at every interaction.
curves_xr = media_effects_kpi.response_curves_data(confidence_level=0.90)
print(curves_xr)

expected = {"spend", "incremental_outcome"}
if not expected.issubset(set(curves_xr.data_vars)):
    raise ValueError(f"Unexpected Meridian response dataset: {list(curves_xr.data_vars)}")
channel_coord = next((key for key in ("channel", "media") if key in curves_xr.coords), None)
if channel_coord is None:
    raise ValueError(f"Expected 'channel' or 'media' coordinate. Available: {list(curves_xr.coords)}")

channel_names = [str(x) for x in curves_xr.coords[channel_coord].values]
print("Paid channels:", channel_names)

def _curve_values(channel, var, metric="mean"):
    da = curves_xr[var].sel({channel_coord: channel})
    if "metric" in da.dims:
        da = da.sel(metric=metric)
    return np.asarray(da.values, dtype=float).reshape(-1)

channel_curves = {}
for ch in channel_names:
    spend = _curve_values(ch, "spend")
    mean = _curve_values(ch, "incremental_outcome", "mean")
    lower = _curve_values(ch, "incremental_outcome", "ci_lo")
    upper = _curve_values(ch, "incremental_outcome", "ci_hi")
    # Spend is monotonically increasing in the standard paid-media response curve.
    good = np.isfinite(spend) & np.isfinite(mean) & np.isfinite(lower) & np.isfinite(upper)
    order = np.argsort(spend[good])
    spend, mean, lower, upper = (arr[good][order] for arr in (spend, mean, lower, upper))
    if len(spend) < 3 or not np.all(np.diff(spend) > 0):
        raise ValueError(f"Bad or duplicate spend grid for {ch}.")
    # 'spend_multiplier' == 1 means the historical channel budget.
    if "spend_multiplier" not in curves_xr.coords:
        raise ValueError("Response curves are missing the spend_multiplier coordinate.")
    full_mult = np.asarray(curves_xr.coords["spend_multiplier"].values, dtype=float)
    raw_spend = _curve_values(ch, "spend")
    historical = float(np.interp(1.0, full_mult, raw_spend))
    channel_curves[ch] = dict(spend=spend, mean=mean, lo=lower, hi=upper, historical=historical)
    if not (spend[0] <= historical <= spend[-1]):
        raise ValueError(f"Historical spend is outside the curve grid for {ch}.")

display(pd.DataFrame({
    "Channel": channel_names,
    "Historical budget ($)": [channel_curves[c]["historical"] for c in channel_names],
    "Min grid spend ($)": [channel_curves[c]["spend"][0] for c in channel_names],
    "Max grid spend ($)": [channel_curves[c]["spend"][-1] for c in channel_names],
}).style.format({"Historical budget ($)":"${:,.0f}", "Min grid spend ($)":"${:,.0f}", "Max grid spend ($)":"${:,.0f}"}))

### 12b. Interactive spend scenarios (estimated incremental **revenue**)

Each slider shows the budget as a percentage of that channel's original historical spend. A 120% slider means 20% more spend. The charts and table display **incremental revenue**, **not ROI percentages**. The values are interpolated from Meridian's precomputed response-curve posterior means; they are not new observations or a new MCMC fit.

To test whether a *reallocation* is useful, keep total spend approximately constant by moving dollars **between** channels, rather than only increasing spending overall. For a mathematically optimized allocation, use Section 12c.

If widgets are not displayed in your version of JupyterLab, use the fallback function in Section 12d.


In [ ]:
import ipywidgets as widgets

# Percent inputs are editable: drag a slider or type the value in its box.
# Restrict to Meridian's precomputed spend-response curve grid.
sliders = {}
for ch in channel_names:
    d = channel_curves[ch]
    low_pct = max(0, int(np.ceil(100 * d["spend"][0] / d["historical"] / 5) * 5))
    high_pct = int(np.floor(100 * d["spend"][-1] / d["historical"] / 5) * 5)
    if not (low_pct <= 100 <= high_pct):
        raise ValueError(f"Slider grid does not include historical spend for {ch}.")
    sliders[ch] = widgets.IntSlider(
        value=100, min=low_pct, max=high_pct, step=5,
        description=ch, continuous_update=False,
        style={"description_width": "110px"},
        layout=widgets.Layout(width="520px"),
    )

budget_out = widgets.Output()

def show_scenario(*_):
    with budget_out:
        budget_out.clear_output(wait=True)
        fig, axs = plt.subplots(1, len(channel_names), figsize=(5.2 * len(channel_names), 4.2), squeeze=False)
        rows = []
        for ax, ch in zip(axs[0], channel_names):
            c = channel_curves[ch]
            old_spend = c["historical"]
            new_spend = old_spend * sliders[ch].value / 100
            old_kpi = np.interp(old_spend, c["spend"], c["mean"])
            new_kpi = np.interp(new_spend, c["spend"], c["mean"])
            lower = np.interp(new_spend, c["spend"], c["lo"])
            upper = np.interp(new_spend, c["spend"], c["hi"])

            ax.plot(c["spend"], c["mean"], lw=2.2, label="Posterior mean")
            ax.fill_between(c["spend"], c["lo"], c["hi"], alpha=.18, label="90% credible band")
            ax.axvline(old_spend, ls="--", color="gray", lw=1.4, label="Historical spend")
            ax.scatter([new_spend], [new_kpi], s=75, zorder=5, label="New budget")
            ax.set_title(ch)
            ax.set_xlabel("Total media budget ($)")
            ax.set_ylabel("Estimated incremental revenue ($)")
            ax.ticklabel_format(style="plain", axis="both", useOffset=False)
            ax.grid(alpha=.15)

            rows.append({
                "Channel": ch,
                "Previous budget ($)": old_spend,
                "New budget ($)": new_spend,
                "Budget change (%)": (new_spend / old_spend - 1) * 100,
                "Previous incremental revenue ($)": old_kpi,
                "New incremental revenue ($)": new_kpi,
                "Change in incremental revenue ($)": new_kpi - old_kpi,
                "Change in budget ($)": new_spend - old_spend,
                "New KPI 90% CI low": lower,
                "New KPI 90% CI high": upper,
            })

        axs[0, -1].legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()

        scenario = pd.DataFrame(rows)
        display(scenario[[
            "Channel", "Previous budget ($)", "New budget ($)",
            "Budget change (%)", "Change in budget ($)",
            "Previous incremental revenue ($)", "New incremental revenue ($)",
            "Change in incremental revenue ($)"
        ]].style.format({
            "Previous budget ($)": "${:,.0f}", "New budget ($)": "${:,.0f}",
            "Budget change (%)": "{:+.0f}%", "Change in budget ($)": "${:+,.0f}",
            "Previous incremental revenue ($)": "${:,.0f}",
            "New incremental revenue ($)": "${:,.0f}",
            "Change in incremental revenue ($)": "${:+,.0f}",
        }))
        print(f"Total historical spend: ${scenario['Previous budget ($)'].sum():,.0f}")
        print(f"Total proposed spend:   ${scenario['New budget ($)'].sum():,.0f}")
        print(f"Change in expected incremental revenue (sum of channel means): "
              f"${scenario['Change in incremental revenue ($)'].sum():+,.0f}")
        print("Note: the aggregate displayed is a sum of posterior means, not a ")
        print("credible interval for the total change or a forecast of total sales.")

for slider in sliders.values():
    slider.observe(show_scenario, names="value")

reset_button = widgets.Button(description="Reset all to 100%", button_style="")
def reset_budgets(_):
    for slider in sliders.values():
        slider.value = 100
reset_button.on_click(reset_budgets)

controls = widgets.VBox([
    widgets.HTML("<b>Change each channel's advertising budget (% of historical)</b>"),
    *sliders.values(), reset_button,
])
display(controls, budget_out)
show_scenario()

### 12c. Optional: Let Meridian optimize a *total* budget

The sliders above explore a **manually chosen** budget for each channel. The following calls Meridian's **BudgetOptimizer** to recommend an allocation that maximizes posterior-mean incremental KPI (for a conversion-trained model) subject to channel-level constraints.

For the class, keep the **same total budget** and use **±30% per-channel bounds**. You can also enter a different total budget. Optimization results are model-based recommendations, not experimental proof.

In [ ]:
from meridian.analysis import optimizer

historical_total = sum(channel_curves[c]["historical"] for c in channel_names)

budget_input = widgets.FloatText(
    value=float(historical_total), description="Total budget ($)",
    style={"description_width": "125px"}, layout=widgets.Layout(width="340px"),
)
constraint_input = widgets.FloatSlider(
    value=30, min=0, max=100, step=5, description="Channel ± %",
    continuous_update=False, style={"description_width": "125px"},
    layout=widgets.Layout(width="400px"),
)
opt_button = widgets.Button(description="Optimize allocation", button_style="primary")
opt_output = widgets.Output()

def run_optimization(_):
    with opt_output:
        opt_output.clear_output(wait=True)
        if budget_input.value <= 0:
            print("Please enter a positive total budget.")
            return
        print("Optimizing with Meridian... This may take some time.")
        try:
            opt = optimizer.BudgetOptimizer(mmm)
            result = opt.optimize(
                fixed_budget=True,
                budget=float(budget_input.value),
                spend_constraint_lower=constraint_input.value / 100,
                spend_constraint_upper=constraint_input.value / 100,
            )
            display(result.plot_spend_delta())
            display(result.plot_incremental_outcome_delta())
            display(result.plot_budget_allocation())
            result.output_optimization_summary("meridian_budget_optimization.html", ".")
            print("Saved: meridian_budget_optimization.html")
        except Exception as exc:
            print(f"Optimization failed: {exc}")
            raise

opt_button.on_click(run_optimization)
display(widgets.VBox([budget_input, constraint_input, opt_button, opt_output]))

### Student discussion

1. Increasing a budget by 25%: does incremental KPI also rise by 25%? Why or why not?
2. Are **historical average ROI** and **marginal ROI** identical? What would saturation imply?
3. Can we move all money to the channel with the highest current average ROI? What are the assumptions and risks?
4. Can a great-fitting model still produce unreliable causal channel contributions?

**Source:** [Google Meridian MediaEffects](https://developers.google.com/meridian/reference/api/meridian/analysis/visualizer/MediaEffects) and [BudgetOptimizer](https://developers.google.com/meridian/reference/api/meridian/analysis/optimizer/BudgetOptimizer). The code in this notebook has been syntax checked, but Meridian fitting and chart execution must be verified in your local `meridian-mmm` environment.

### 12d. Budget scenarios without Jupyter widgets (reliable fallback)

If JupyterLab does not render `ipywidgets`, run this section instead. It uses the **same posterior spend-response curves already computed in Section 12a**, and does not refit the Bayesian model. Change the three percentages in the last cell and rerun it. **100** is historical spend, **120** is +20%, **80** is -20%.

This is *mean response-curve interpolation* under Meridian's fixed flighting/cost assumptions. The table is not a forecast of total sales, and the interval bands for individual levels are not posterior intervals of the budget changes.


In [ ]:
# Change percentage inputs in the FINAL CELL below; no ipywidgets needed.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def budget_scenario(google=100, meta=100, youtube=100):
    if "channel_curves" not in globals():
        raise RuntimeError("Run section 12a first to compute channel_curves.")
    pct = {"Google Search": google, "Meta": meta, "YouTube": youtube}
    available = list(channel_curves)
    missing = set(available) - set(pct)
    if missing:
        raise ValueError(f"Missing percentages for channels: {missing}")
    fig, axs = plt.subplots(1, len(available), figsize=(5.2 * len(available), 4.2), squeeze=False)
    rows = []
    for ax, channel in zip(axs[0], available):
        curve = channel_curves[channel]
        old_spend = curve["historical"]
        new_spend = old_spend * float(pct[channel]) / 100
        if not (curve["spend"][0] <= new_spend <= curve["spend"][-1]):
            raise ValueError(f"{channel}: {pct[channel]}% is outside the precomputed response curve range.")
        old_kpi = float(np.interp(old_spend, curve["spend"], curve["mean"]))
        new_kpi = float(np.interp(new_spend, curve["spend"], curve["mean"]))
        ax.plot(curve["spend"], curve["mean"], label="Posterior mean", linewidth=2)
        ax.fill_between(curve["spend"], curve["lo"], curve["hi"], alpha=.18, label="90% credible band")
        ax.axvline(old_spend, color="gray", linestyle="--", label="Historical spend")
        ax.scatter(new_spend, new_kpi, s=70, zorder=5, label="Proposed spend")
        ax.set_title(channel)
        ax.set_xlabel("Budget ($)")
        ax.set_ylabel("Incremental revenue ($)")
        ax.grid(alpha=.15)
        ax.ticklabel_format(axis="both", style="plain", useOffset=False)
        rows.append({
            "Channel": channel,
            "Previous budget ($)": old_spend,
            "New budget ($)": new_spend,
            "Budget change (%)": 100 * (new_spend / old_spend - 1),
            "Previous incremental revenue ($)": old_kpi,
            "New incremental revenue ($)": new_kpi,
            "Change in incremental revenue ($)": new_kpi - old_kpi,
            "Change in budget ($)": new_spend - old_spend,
        })
    axs[0, -1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    table = pd.DataFrame(rows)
    display(table.style.format({
        "Previous budget ($)": "${:,.0f}", "New budget ($)": "${:,.0f}",
        "Budget change (%)": "{:+.0f}%",
        "Previous incremental revenue ($)": "${:,.0f}",
        "New incremental revenue ($)": "${:,.0f}",
        "Change in incremental revenue ($)": "${:+,.0f}",
        "Change in budget ($)": "${:+,.0f}",
    }))
    print(f"Total historical budget: ${table['Previous budget ($)'].sum():,.0f}")
    print(f"Total proposed budget:   ${table['New budget ($)'].sum():,.0f}")
    print(f"Change in expected incremental revenue: ${table['Change in incremental revenue ($)'].sum():+,.0f}")
    return table


In [ ]:
# Edit the three percentages and rerun. No widget extension required.
# This is an EXAMPLE scenario with a larger total budget, not an optimal allocation.
scenario_table = budget_scenario(
    google=100,
    meta=100,
    youtube=100,
)
